In [1]:
import os
import glob
import json
import re
import pandas as pd
import numpy as np

# ==================== 1. 設定 INTRA 實驗資料夾對照表 ====================
INTRA_EXPERIMENT_CONFIGS = [
    # --- 1. INTRA 實驗組：有權威 + Memory/Reflection ---
    {
        "folder": "social-simulate-v3-agent-qwen3-authority-intra-result",
        "family": "Qwen3 Family (INTRA)",
        "condition": "Authority + Memory/Reflection",
        "has_authority": True
    },
    # --- 2. INTRA 控制組：無權威 + Memory/Reflection ---
    {
        "folder": "social-simulate-v3-agent-qwen3-control-intra-result",
        "family": "Qwen3 Family (INTRA)",
        "condition": "Control (No Authority) + Memory/Reflection",
        "has_authority": False
    }
]

# 定義要追蹤的模型規模清單（自動清洗名稱，如 qwen3:8b -> 8B）
MODEL_SCALE_MAP = {
    "qwen3:8b": "8B",
    "qwen3:14b": "14B",
    "qwen3:30b": "30B"
}

def clean_model_name(raw_name):
    """將 qwen3:8b 等原生名稱轉換為論文簡潔標籤 8B/14B/30B"""
    raw_lower = str(raw_name).lower()
    for key, label in MODEL_SCALE_MAP.items():
        if key in raw_lower:
            return label
    # 自動用正則抓取參數量
    match = re.search(r'(\d+b)', raw_lower)
    return match.group(1).upper() if match else raw_name


def parse_intra_run_details(df):
    """
    深度拆解每一場 Run 的 agent_details_json，提煉出微觀層次（按 8B/14B/30B）的行為數據
    """
    micro_records = []
    macro_auth_success = []

    for run_idx, row in df.iterrows():
        json_str = row.get("agent_details_json", "{}")
        try:
            details = json.loads(json_str)
        except Exception:
            continue

        # 找出本場 Run 的權威專家資訊
        auth_name = None
        auth_init_stance = None
        auth_model_scale = None

        for name, info in details.items():
            if info.get("is_authority", False):
                auth_name = name
                auth_init_stance = info.get("initial_stance", "")
                auth_model_scale = clean_model_name(info.get("model_name", "Unknown"))
                break

        swayed_count = 0
        non_auth_count = 0

        # 遍歷場內每個 Agent，紀錄微觀指標
        for name, info in details.items():
            is_auth = info.get("is_authority", False)
            model_scale = clean_model_name(info.get("model_name", "Unknown"))
            init_s = float(info.get("initial_score", 5.0))
            final_s = float(info.get("final_score", 5.0))
            shift = final_s - init_s
            shift_type = info.get("shift_type", "堅持")

            # 檢定是否被權威引導（向權威靠攏）
            swayed_towards_auth = False
            if auth_name and not is_auth:
                non_auth_count += 1
                if auth_init_stance == "Support" and shift > 0:
                    swayed_towards_auth = True
                    swayed_count += 1
                elif auth_init_stance == "Oppose" and shift < 0:
                    swayed_towards_auth = True
                    swayed_count += 1

            micro_records.append({
                "run_id": row.get("run_id", run_idx + 1),
                "agent_name": name,
                "model_scale": model_scale,
                "is_authority": is_auth,
                "initial_score": init_s,
                "final_score": final_s,
                "stance_shift": shift,
                "shift_type": shift_type,
                "swayed_towards_auth": swayed_towards_auth if not is_auth else np.nan,
                "auth_model_scale_of_run": auth_model_scale
            })

        if non_auth_count > 0:
            macro_auth_success.append(swayed_count / non_auth_count)

    return pd.DataFrame(micro_records), (np.mean(macro_auth_success) * 100 if macro_auth_success else np.nan)


def process_intra_folder(folder_path, has_authority):
    """
    處理單一 INTRA 實驗資料夾，兼顧「巨觀指標」與「微觀模型規模拆解」
    """
    csv_pattern = os.path.join(folder_path, "*_metrics.csv")
    csv_files = glob.glob(csv_pattern)

    if not csv_files:
        print(f"⚠️ 警告：在路徑 '{folder_path}' 中找不到 *_metrics.csv 檔案。")
        return None, None

    target_csv = sorted(csv_files)[-1]
    df = pd.read_csv(target_csv)

    # 1. 拆解微觀 Agent 細節數據
    micro_df, macro_auth_rate = parse_intra_run_details(df)

    # 2. 彙整巨觀指標 (Macro Metrics)
    macro_summary = {
        "runs_count": len(df),
        "mean_shift": round(df["mean_shift"].mean(), 3) if "mean_shift" in df else np.nan,
        "polarization_index": round(df["polarization_index"].mean(), 3) if "polarization_index" in df else np.nan,
        "fragmentation_index": round(df["fragmentation_index"].mean(), 3) if "fragmentation_index" in df else np.nan,
        "conformity_rate": round(df["conformity_rate"].mean(), 3) if "conformity_rate" in df else np.nan,
        "mean_authority_in_degree": round(df["authority_in_degree"].mean(), 2) if "authority_in_degree" in df else 0.0,
        "total_authority_in_degree": int(df["authority_in_degree"].sum()) if "authority_in_degree" in df else 0,
        "mean_cascade_depth": round(df["cascade_depth"].mean(), 2) if "cascade_depth" in df else 0.0,
        "max_cascade_depth": int(df["cascade_depth"].max()) if "cascade_depth" in df else 0,
        "authority_success_rate": round(macro_auth_rate, 2) if (has_authority and not np.isnan(macro_auth_rate)) else "N/A (控制組)"
    }

    # 3. 按模型規模（8B vs 14B vs 30B）進行微觀交叉彙整 (Micro Breakdown)
    micro_breakdown_list = []
    if micro_df is not None and not micro_df.empty:
        for scale in ["8B", "14B", "30B"]:
            sub_df = micro_df[micro_df["model_scale"] == scale]
            non_auth_sub = sub_df[sub_df["is_authority"] == False]

            if not sub_df.empty:
                avg_shift = round(sub_df["stance_shift"].mean(), 3)
                # 計算該規模模型被順服/同化的比例
                sway_rate = round(non_auth_sub["swayed_towards_auth"].mean() * 100, 2) if (has_authority and not non_auth_sub.empty) else "N/A"
                # 計算堅持立場的比例
                persist_rate = round((sub_df["shift_type"] == "堅持").mean() * 100, 2)

                # 計算當該規模模型「擔綱權威」時的引導成功率
                auth_role_df = micro_df[micro_df["auth_model_scale_of_run"] == scale]
                auth_lead_success = round(auth_role_df[auth_role_df["is_authority"] == False]["swayed_towards_auth"].mean() * 100, 2) if (has_authority and not auth_role_df.empty) else "N/A"

                micro_breakdown_list.append({
                    "Model_Scale": scale,
                    "Mean_Stance_Shift": avg_shift,
                    "Persistence_Rate(%)": persist_rate,
                    "Swayed_By_Authority_Rate(%)": sway_rate,
                    "Lead_Success_When_Authority(%)": auth_lead_success
                })

    return macro_summary, pd.DataFrame(micro_breakdown_list)


def main():
    print("=" * 90)
    print(" 🚀 開始執行 INTRA (家族內多模型混編 8B/14B/30B) 實驗數據整合分析")
    print("=" * 90 + "\n")

    macro_results = []
    micro_all_results = []

    for config in INTRA_EXPERIMENT_CONFIGS:
        folder = config["folder"]
        family = config["family"]
        condition = config["condition"]
        has_auth = config["has_authority"]

        print(f"🔍 正在處理：[{family}] - [{condition}]...")
        macro_sum, micro_df = process_intra_folder(folder, has_auth)

        if macro_sum:
            macro_sum["Family"] = family
            macro_sum["Condition"] = condition
            macro_results.append(macro_sum)

        if micro_df is not None and not micro_df.empty:
            micro_df["Condition"] = condition
            micro_all_results.append(micro_df)

    # ------------------ 輸出 1：巨觀全場指標矩陣 ------------------
    if macro_results:
        macro_df = pd.DataFrame(macro_results)
        cols = ["Family", "Condition", "runs_count", "mean_shift", "polarization_index", 
                "fragmentation_index", "conformity_rate", "mean_authority_in_degree", 
                "mean_cascade_depth", "authority_success_rate"]
        macro_df = macro_df[cols].rename(columns={
            "Family": "模型家族", "Condition": "實驗條件", "runs_count": "總模擬次數",
            "mean_shift": "平均立場位移", "polarization_index": "群體極化指數",
            "fragmentation_index": "群體分裂指數", "conformity_rate": "從眾率",
            "mean_authority_in_degree": "權威次均被提及數",
            "mean_cascade_depth": "級聯擴散平均深度", "authority_success_rate": "權威總體引導成功率(%)"
        })

        print("\n" + "=" * 100)
        print("📊 表 1：INTRA 模式巨觀總體對比表 (Macro Discussion Matrix)")
        print("=" * 100)
        print(macro_df.to_string(index=False))

        macro_df.to_csv("intra_macro_summary_matrix.csv", index=False, encoding="utf-8-sig")
        print("\n✅ 巨觀總體分析已匯出至：intra_macro_summary_matrix.csv")

    # ------------------ 輸出 2：微觀模型規模拆解矩陣 ------------------
    if micro_all_results:
        combined_micro_df = pd.concat(micro_all_results, ignore_index=True)
        cols_micro = ["Condition", "Model_Scale", "Mean_Stance_Shift", "Persistence_Rate(%)", 
                      "Swayed_By_Authority_Rate(%)", "Lead_Success_When_Authority(%)"]
        combined_micro_df = combined_micro_df[cols_micro].rename(columns={
            "Condition": "實驗條件",
            "Model_Scale": "模型規模",
            "Mean_Stance_Shift": "平均立場位移",
            "Persistence_Rate(%)": "立場堅持率(%)",
            "Swayed_By_Authority_Rate(%)": "被權威同化順服率(%)",
            "Lead_Success_When_Authority(%)": "當該規模擔任權威時之引導成功率(%)"
        })

        print("\n" + "=" * 100)
        print("🔬 表 2：INTRA 模式微觀模型規模拆解表 (Micro Model-Scale Breakdown Matrix)")
        print("=" * 100)
        print(combined_micro_df.to_string(index=False))

        combined_micro_df.to_csv("intra_micro_scale_breakdown.csv", index=False, encoding="utf-8-sig")
        print("✅ 微觀規模拆解分析已匯出至：intra_micro_scale_breakdown.csv")


if __name__ == "__main__":
    main()

 🚀 開始執行 INTRA (家族內多模型混編 8B/14B/30B) 實驗數據整合分析

🔍 正在處理：[Qwen3 Family (INTRA)] - [Authority + Memory/Reflection]...
🔍 正在處理：[Qwen3 Family (INTRA)] - [Control (No Authority) + Memory/Reflection]...

📊 表 1：INTRA 模式巨觀總體對比表 (Macro Discussion Matrix)
                模型家族                                       實驗條件  總模擬次數  平均立場位移  群體極化指數  群體分裂指數   從眾率  權威次均被提及數  級聯擴散平均深度 權威總體引導成功率(%)
Qwen3 Family (INTRA)              Authority + Memory/Reflection     25  -0.659   0.460   0.694 0.318     28.68      2.56         51.2
Qwen3 Family (INTRA) Control (No Authority) + Memory/Reflection     25  -0.820   0.127   0.639 0.446      0.00      0.00    N/A (控制組)

✅ 巨觀總體分析已匯出至：intra_macro_summary_matrix.csv

🔬 表 2：INTRA 模式微觀模型規模拆解表 (Micro Model-Scale Breakdown Matrix)
                                      實驗條件 模型規模  平均立場位移  立場堅持率(%) 被權威同化順服率(%) 當該規模擔任權威時之引導成功率(%)
             Authority + Memory/Reflection   8B   -0.90      34.0       68.29              35.56
             Authority + Memory/Reflection  14B   -1.08